# HPO

El proceso parte del dataset preprocesado, carga las variables candidatas y aplica la selección específica del experimento. Después separa `X`, el target residual y el segmento judicial, manteniendo el segmento fuera de las variables del modelo para evitar leakage. Las variables categóricas se convierten al tipo nativo esperado por XGBoost y se revisa la distribución del target.

La muestra se estratifica por `seg_PJ_first_judicial_procedure_type_id_corrected`. Primero se reserva un holdout final que no participa en el HPO; sobre el resto se construyen folds estratificados para comparar todos los trials bajo las mismas particiones.

Cada trial propone hiperparámetros de `XGBRegressor` y de ponderación de observaciones (`w_scale`, `w_power`), dando más peso a targets alejados de 0.5. Después entrena un modelo por fold y calcula MAE de entrenamiento y validación. El objetivo combina el MAE global de validación con el promedio no ponderado del MAE de cada segmento: `GLOBAL_MAE_WEIGHT * MAE_global + (1 - GLOBAL_MAE_WEIGHT) * MAE_segmentos`. Así se conserva el rendimiento general y se evita que los segmentos con más observaciones dominen por completo la búsqueda.

Optuna usa TPE para explorar el espacio, poda trials poco prometedores con el objetivo combinado parcial y controla el sobreajuste mediante la restricción `MAE_validación_global - MAE_entrenamiento <= TAU`. Los trials completos se ordenan por el objetivo combinado, conservando por separado las métricas globales, por segmento y de sobreajuste.

Finalmente se guardan el estudio y sus resultados, se recuperan los mejores trials factibles, se reentrenan con toda la muestra de HPO y se evalúan una única vez sobre el holdout. Las métricas, predicciones, gráficos y parámetros resultantes se comparan y se persisten como artefactos del experimento.

In [ ]:
#%pip install "enki-aws>=0.1.10"
#%pip install "pandas>=1.5.1"
#%pip install "numpy<2.0.0"
#%pip install "awswrangler"
#%pip install "seaborn"
#%pip install "xgboost"
#%pip install "shap"
#%pip install "scipy"
#%pip install "scikit-learn"
#%pip install "optuna==3.6.1"

# Imports

In [ ]:
import importlib
import importlib.util
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from time import time
import joblib

import awswrangler as wr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from scipy import stats
from sklearn.base import clone
from sklearn.metrics import get_scorer, mean_absolute_error
from sklearn.model_selection import (
    StratifiedShuffleSplit,
    cross_val_predict,
    StratifiedKFold
)
from xgboost import XGBRegressor
from enki import GenericConnector

import optuna
from optuna import Trial
from optuna.importance import get_param_importances
#from optbinning import OptimalBinning

from optuna.samplers import TPESampler
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, mean_absolute_percentage_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
import logging

optuna.logging.set_verbosity(optuna.logging.INFO)

logger = logging.getLogger("optuna")
logger.handlers.clear()

file_handler = logging.FileHandler("optuna.log")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
)

logger.addHandler(file_handler)
logger.propagate = False

In [ ]:
_progreso_dir = (
    Path.home()
    / "etltrainjudicializ"
    / "notebooks"
    / "03_modelos"
    / "progreso"
    / "pf"
    / "m1"
    / "experiments"
)

if str(_progreso_dir) not in sys.path:
    sys.path.insert(0, str(_progreso_dir))

import evals_regression
importlib.reload(evals_regression)

evals_regression = importlib.reload(evals_regression)

from evals_regression import summarize_model_results, add_real_vs_pred_pages_to_pdf

# Env vars

In [ ]:
hpo_name="v5_opt_segments_5ktrials"

In [ ]:
debug = True

In [ ]:
person_type = "pf"
param_days_duration_limit=559

PATH_PROJECT = "/data/sandboxes/ranh/data/proyectos/cobranzas/modelo_judicializacion/03_modelos/"
READ_PATH_DATA = PATH_PROJECT+f"data/{param_days_duration_limit}/{person_type}"
WRITE_PATH = PATH_PROJECT + f"progreso/{person_type}/m1/experiments_v2/base_models/hpo/"

bucket = os.getenv("SANDBOX_BUCKET_DATA")
s3_data = "s3://" + bucket

In [ ]:
gen_connector = GenericConnector()
fs = gen_connector._get_fs()

In [ ]:
date_col = "PJ_procedure_regist_date_month"
target_col = "target_progress_score"
segment_col = "PJ_first_judicial_procedure_type_id_corrected"
group_cols = [
    segment_col,
    "agencia",
]

# Functions

In [ ]:
def load_var_from_py(py_path, var_name):
    py_path = Path(py_path)

    if not py_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {py_path}")

    spec = importlib.util.spec_from_file_location(py_path.stem, py_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, var_name):
        raise AttributeError(f"El archivo {py_path} no contiene la variable {var_name}")

    return list(getattr(module, var_name))

# Load data

## Load variables

In [ ]:
vars_path = (
    Path.home()
    / "etltrainjudicializ"
    / "notebooks"
    / "03_modelos"
    / "progreso"
    / person_type
    / "m1"
    / "experiments_v2"
    / "base_models"
    / "correlation"
    / "vars_surviving_phase1.py"
)

selected_features = load_var_from_py(
    py_path=vars_path,
    var_name="VARS_SURVIVING_PHASE1",
)


print("selected_features:", len(selected_features))

In [ ]:
vars_key = ["PJ_id",date_col, target_col]

## add extra features

In [ ]:
new_features = [
    "mora_antiguedad_dias",
    "mora_deuda_principal_mora",
]

selected_features = selected_features + [
    f for f in new_features if f not in selected_features
]

selected_features=sorted(selected_features)

## Load data

In [ ]:
df = pd.read_parquet(path=f"{bucket}{READ_PATH_DATA}/train_features",
                     filesystem=fs,
                     columns=list(set(vars_key) | set(selected_features) | set(group_cols))
                  )

print("df:",df.shape)

# Parameters

In [ ]:
SEED = 42

# Metrica y opciones de busqueda
SCORING = "neg_mean_absolute_error"

# Cross-validation (unico para todas las fases)
K_FOLDS = 4
N_REPEATS = 8
VALIDATION_SIZE = 0.15


MODEL_OBJECTIVE_OPTIONS = ["reg:logistic"]
SEED_OPTIONS = [42]
EVAL_METRIC     = "mae"            # cosmetico sin early stopping
GLOBAL_MAE_WEIGHT = 0.5  # peso del MAE global en el objetivo combinado


SUCCESS_THRESHOLD = 0.15  # abs(y_true - y_pred) <= threshold

In [ ]:
# Optuna
N_TRIALS = 5000
TAU = 0.02  # restriccion directa: gap_overfitting <= TAU

N_STARTUP_TRIALS = int(min(max(N_TRIALS * 0.05, 25), 250))
N_PRUNER_STARTUP_TRIALS = N_STARTUP_TRIALS
N_WARMUP_STEPS = int(max(2, K_FOLDS * 0.5))
N_MIN_TRIALS = int(min(max(N_TRIALS * 0.01, 5), 50))

print(f"N_TRIALS                 = {N_TRIALS}")
print(f"K_FOLDS                  = {K_FOLDS}")
print(f"N_STARTUP_TRIALS         = {N_STARTUP_TRIALS}")
print(f"N_PRUNER_STARTUP_TRIALS  = {N_PRUNER_STARTUP_TRIALS}")
print(f"N_WARMUP_STEPS           = {N_WARMUP_STEPS}")
print(f"N_MIN_TRIALS             = {N_MIN_TRIALS}")

# Sample definition

In [ ]:
X_train = df.loc[:, selected_features].copy()
y_train = df[target_col].astype(float).copy()

print(X_train.shape)
print(y_train.shape)

In [ ]:
# =========================================================
# Categóricas nativas para XGBoost
# Solo aplica si existen columnas object
# =========================================================
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

if categorical_cols:
    print(f"Columnas object convertidas a category: {len(categorical_cols)}")
    print(categorical_cols)

    for col in categorical_cols:
        X_train[col] = X_train[col].astype("category")
else:
    print("No hay columnas object en X_train. No se convierte nada a category.")


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# Check target

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ==========================
# Preparación
# ==========================
y = df[target_col].dropna()

assert len(y) > 0, f"{target_col} no contiene valores válidos."

# ==========================
# Estadísticos
# ==========================
print("=" * 80)
print(f"Variable: {target_col}")
print("=" * 80)

print(y.describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))

print("\nSkewness :", round(y.skew(), 4))
print("Kurtosis :", round(y.kurtosis(), 4))

# ==========================
# Gráficos
# ==========================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma
y.hist(
    bins=60,
    ax=axes[0, 0],
    density=True,
    alpha=0.6,
)

y.plot.kde(ax=axes[0, 0], lw=2)

axes[0, 0].set_title("Distribución")
axes[0, 0].set_xlabel(target_col)
axes[0, 0].grid(alpha=0.3)

# Boxplot
axes[0, 1].boxplot(y, vert=False)
axes[0, 1].set_title("Boxplot")
axes[0, 1].set_xlabel(target_col)

# QQ plot
stats.probplot(y, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ Plot")

# Histograma log1p (si procede)
if (y >= 0).all():
    y_log = np.log1p(y)

    y_log.hist(
        bins=60,
        ax=axes[1, 1],
        density=True,
        alpha=0.6,
    )
    y_log.plot.kde(ax=axes[1, 1], lw=2)

    axes[1, 1].set_title("Distribución log1p(target)")
    axes[1, 1].grid(alpha=0.3)
else:
    axes[1, 1].text(
        0.5,
        0.5,
        "log1p no aplicable\n(valores negativos)",
        ha="center",
        va="center",
        fontsize=12,
    )
    axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()

# Hold-out split for HPO

In [ ]:
# =========================================================
# Estratificación por segmentos
# =========================================================
strat = (
    df.loc[:, group_cols]
    .astype("object")
    .astype(str)
    .agg("||".join, axis=1)
)

In [ ]:
if debug:
    
    strat_counts = strat.value_counts()
    
    print("n_estratos:", strat_counts.shape[0])
    print(f"estratos con < {K_FOLDS} muestras:", strat_counts.lt(K_FOLDS).sum())
    
    assert strat_counts.min() >= K_FOLDS, (
        f"Hay estratos con menos de {K_FOLDS} muestras. "
        "Agrupa estratos raros antes de usar StratifiedKFold."
    )

In [ ]:
# Hold-out split para HPO y validacion final
validation_splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
)

hpo_train_idx, validation_idx = next(validation_splitter.split(X_train, strat))

X_hpo_train = X_train.iloc[hpo_train_idx].copy()
y_hpo_train = y_train.iloc[hpo_train_idx].copy()
segments_hpo_train = df[segment_col].iloc[hpo_train_idx].copy()
strat_hpo_train = strat.iloc[hpo_train_idx].copy()

X_validation_holdout = X_train.iloc[validation_idx].copy()
y_validation_holdout = y_train.iloc[validation_idx].copy()
strat_validation_holdout = strat.iloc[validation_idx].copy()

if debug:
    strat_hpo_counts = strat_hpo_train.value_counts()
    assert strat_hpo_counts.min() >= K_FOLDS, (
        f"Hay estratos con menos de {K_FOLDS} muestras en train HPO. "
        "Ajusta VALIDATION_SIZE o agrupa estratos raros."
    )

print("X_hpo_train:", X_hpo_train.shape)
print("X_validation_holdout:", X_validation_holdout.shape)

# CV definition for HPO

In [ ]:
cv_optuna = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=SEED,
)

splits_optuna = list(cv_optuna.split(X_hpo_train, strat_hpo_train))

print(f"CV Optuna: {K_FOLDS} folds estratificados sobre train HPO, sin repeats")

## Optuna Search Space

In [ ]:
def get_optuna_params(
    trial,
    model_objective_options=MODEL_OBJECTIVE_OPTIONS,
    eval_metric="mae",
    seed_options=SEED_OPTIONS,
    use_sample_weight=False,
):
    model_objective = trial.suggest_categorical("objective", model_objective_options)
    seed = trial.suggest_categorical("random_state", seed_options)

    params = {
        # Objetivo del modelo
        "objective": model_objective,
        "eval_metric": eval_metric,

        # Backend
        "tree_method": "hist",
        "enable_categorical": True,

        # Boosting
        "n_estimators": trial.suggest_int("n_estimators", 1000, 6000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.08, log=True),

        # Complejidad
        "max_depth": trial.suggest_int("max_depth", 5, 9, step=1),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 12, step=1),
        "gamma": trial.suggest_float("gamma", 0.0, 3.0, step=0.1),

        # Muestreo
        "subsample": trial.suggest_float("subsample", 0.5, 0.9, step=0.05),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9, step=0.05),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.9, step=0.05),

        # Regularización
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 100.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),

        # Reproducibilidad / cómputo
        "random_state": seed,
        "n_jobs": -1,
        "verbosity": 0,
    }

    weight_params = None
    if use_sample_weight:
        weight_params = {
            "w_scale": trial.suggest_float("w_scale", 0.0, 5.0),
            "w_power": trial.suggest_float("w_power", 0.5, 3.0),
        }

    return params, weight_params

## Optuna Objective

In [ ]:
def objective(
    trial,
    X_train,
    y_train,
    segments,
    cv,
    eval_metric,
    global_mae_weight,
    tau=None,
    monotone_constraints=None,
    use_sample_weight=False,
):
    params, weight_params = get_optuna_params(
        trial,
        eval_metric=eval_metric,
        use_sample_weight=use_sample_weight,
    )

    if monotone_constraints is not None:
        params["monotone_constraints"] = monotone_constraints

    train_mae_scores = []
    val_mae_scores = []
    val_segment_mae_scores = []
    val_objective_scores = []
    segment_values = segments.dropna().unique().tolist()

    for fold_idx, (train_idx, val_idx) in enumerate(cv, start=1):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]
        segments_val = segments.iloc[val_idx]

        fit_kwargs = {"verbose": False}
        if use_sample_weight:
            target_distance = np.abs(y_tr.to_numpy() - 0.5)
            fit_kwargs["sample_weight"] = (
                1
                + weight_params["w_scale"]
                * target_distance ** weight_params["w_power"]
            )

        model = XGBRegressor(**params)

        model.fit(
            X_tr,
            y_tr,
            **fit_kwargs,
        )

        pred_tr = model.predict(X_tr)
        pred_val = model.predict(X_val)

        train_mae = mean_absolute_error(y_tr, pred_tr)
        val_mae = mean_absolute_error(y_val, pred_val)

        maes_by_segment = {}
        for segment in segment_values:
            mask = segments_val.eq(segment).fillna(False).to_numpy(dtype=bool)
            if mask.any():
                maes_by_segment[segment] = mean_absolute_error(
                    y_val.iloc[mask],
                    pred_val[mask],
                )

        val_segment_mae = float(np.mean(list(maes_by_segment.values())))
        val_objective = (
            global_mae_weight * val_mae
            + (1 - global_mae_weight) * val_segment_mae
        )

        train_mae_scores.append(train_mae)
        val_mae_scores.append(val_mae)
        val_segment_mae_scores.append(val_segment_mae)
        val_objective_scores.append(val_objective)

        # Objetivo combinado parcial para pruning
        trial.report(np.mean(val_objective_scores), step=fold_idx)

        if trial.should_prune():
            raise optuna.TrialPruned()

    train_mae_mean = float(np.mean(train_mae_scores))
    val_mae_mean = float(np.mean(val_mae_scores))
    val_segment_mae_mean = float(np.mean(val_segment_mae_scores))
    val_objective_mean = float(np.mean(val_objective_scores))
    val_mae_std = float(np.std(val_mae_scores, ddof=1))
    train_mae_std = float(np.std(train_mae_scores, ddof=1))

    gap_overfitting = float(val_mae_mean - train_mae_mean)

    trial.set_user_attr("train_mae_mean", train_mae_mean)
    trial.set_user_attr("val_mae_mean", val_mae_mean)
    trial.set_user_attr("val_segment_mae_mean", val_segment_mae_mean)
    trial.set_user_attr("global_mae_weight", global_mae_weight)
    trial.set_user_attr("val_mae_std", val_mae_std)
    trial.set_user_attr("train_mae_std", train_mae_std)
    trial.set_user_attr("gap_overfitting", gap_overfitting)

    # Restriccion directa: factible si gap - tau <= 0
    if tau is not None:
        trial.set_user_attr("constraint", (gap_overfitting - tau,))

    return val_objective_mean

## Optuna Study (run)

In [ ]:
def constraints_func(trial):
    # Trials podados no alcanzan a guardar 'constraint' -> infactibles (baja prioridad)
    return trial.user_attrs.get("constraint", (1.0,))

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=SEED,
    constraints_func=constraints_func,
    n_startup_trials=N_STARTUP_TRIALS,
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=N_PRUNER_STARTUP_TRIALS,
    n_warmup_steps=N_WARMUP_STEPS,
    n_min_trials=N_MIN_TRIALS,
)

study = optuna.create_study(
    study_name="xgb_hpo_m1",
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
)

study.optimize(
    lambda trial: objective(
        trial=trial,
        X_train=X_hpo_train,
        y_train=y_hpo_train,
        segments=segments_hpo_train,
        cv=splits_optuna,
        monotone_constraints=None,
        eval_metric=EVAL_METRIC,
        global_mae_weight=GLOBAL_MAE_WEIGHT,
        tau=TAU,
        use_sample_weight=True,
    ),
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)


## Results

In [ ]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs", "state")
)

# =========================================================
# Tabla ordenada de trials completos
# =========================================================
first_cols = [
    "rank",
    "number",
    "objective_mean",
    "val_mae_mean",
    "val_segment_mae_mean",
    "train_mae_mean",
    "val_mae_std",
    "train_mae_std",
    "gap_overfitting",
]

trials_results = (
    trials_df
    .query("state == 'COMPLETE'")
    .rename(columns={
        "value": "objective_mean",
        "user_attrs_train_mae_mean": "train_mae_mean",
        "user_attrs_val_mae_mean": "val_mae_mean",
        "user_attrs_val_segment_mae_mean": "val_segment_mae_mean",
        "user_attrs_val_mae_std": "val_mae_std",
        "user_attrs_train_mae_std": "train_mae_std",
        "user_attrs_gap_overfitting": "gap_overfitting",
    })
    .sort_values("objective_mean", ascending=True)
    .reset_index(drop=True)
)

# Rank según el orden ya calculado
trials_results.insert(0, "rank", trials_results.index + 1)

# Reordenar columnas: métricas principales primero, resto después
remaining_cols = [
    col for col in trials_results.columns
    if col not in first_cols
]

trials_results = trials_results[first_cols + remaining_cols]
trials_results.drop(columns=["state"], inplace=True)

trials_results["constraint_gap"] = trials_results["gap_overfitting"] - TAU
trials_results["flag_feasible_tau"] = trials_results["constraint_gap"] <= 0

print("Top 5:")
display(trials_results.head(5))

In [ ]:
if debug:
    cols = [
        "rank",
        "number", 
        "objective_mean",
        "val_mae_mean",
        "val_segment_mae_mean",
        "train_mae_mean",
        "gap_overfitting",
        "constraint_gap",
        "params_min_child_weight",
        "params_max_depth",
        "params_n_estimators",
       "params_objective"
    ]
    print(f"Top 10 resultados que cumplen con la restriccion de TAU={TAU} (overfitting maximo permitido)")
    display(
        trials_results
        .query("flag_feasible_tau")
        .sort_values("objective_mean", ascending=True)
        .loc[:, cols]
        .reset_index(drop=True)
        .head(10)
    )

## Plot

In [ ]:
# =========================================================
# Best trial según menor objetivo combinado
# =========================================================
best_row = trials_results.loc[trials_results["rank"].eq(1)].iloc[0]

# =========================================================
# Plot objetivo combinado vs gap overfitting
# =========================================================
fig, ax = plt.subplots(figsize=(10, 6))

# Todos los trials
ax.scatter(
    trials_results["objective_mean"],
    trials_results["gap_overfitting"],
    alpha=0.7,
    label="Trials completos",
)

# Best trial
ax.scatter(
    best_row["objective_mean"],
    best_row["gap_overfitting"],
    color="red",
    #s=120,
    label=f"Best trial #{int(best_row['number'])}",
    zorder=5,
)

# Línea opcional si quieres visualizar gap = 0
ax.axhline(
    y=0,
    linestyle="--",
    linewidth=1,
    alpha=0.7,
)

ax.set_xlabel("Objetivo combinado de validación")
ax.set_ylabel("Gap overfitting: val_mae - train_mae")
ax.set_title("Optuna trials: objetivo combinado vs overfitting gap")
ax.grid(True)
ax.legend()

plt.tight_layout()

# =========================================================
# Guardar en PDF
# =========================================================
output_path = "optuna_objective_vs_gap_overfitting.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight")

plt.show()

print(f"Plot guardado en: {output_path}")

In [ ]:
fig = optuna.visualization.plot_pareto_front(
    study,
    targets=lambda t: (
        t.value,
        t.user_attrs["gap_overfitting"],
    ),
    target_names=[
        "Objetivo combinado de validación",
        "Gap overfitting",
    ],
)

fig.write_html(
    "pareto_front.html",
    include_plotlyjs="cdn",  # o True para que sea autocontenido
)

fig.show()

## Write

In [ ]:
study_json = {
    "study_name": study.study_name,
    "direction": study.direction.name,
    "best_params": study.best_params,
    "best_value": study.best_value,
    "best_trial": study.best_trial.number,
    "trials": [
        {
            "number": t.number,
            "state": t.state.name,
            "value": t.value,
            "params": t.params,
            "distributions": {
                k: str(v) for k, v in t.distributions.items()
            },
            "user_attrs": t.user_attrs,
            "system_attrs": t.system_attrs,
            "datetime_start": (
                t.datetime_start.isoformat()
                if t.datetime_start else None
            ),
            "datetime_complete": (
                t.datetime_complete.isoformat()
                if t.datetime_complete else None
            ),
        }
        for t in study.trials
    ],
}

with open("study.json", "w") as f:
    json.dump(study_json, f, indent=4)

wr.s3.upload(local_file=f"study.json", path=s3_data + WRITE_PATH + f"{hpo_name}/study.json")

In [ ]:
joblib.dump(
    study,
    f"study.pkl",
    compress=1
)

wr.s3.upload(local_file=f"study.pkl", path=s3_data + WRITE_PATH + f"{hpo_name}/study.pkl")

gen_connector.write_file(
    trials_results,
    target=WRITE_PATH + f"{hpo_name}/trials_results.csv",
    format_type="csv",
    index=False,
)

wr.s3.upload(local_file=f"optuna.log", path=s3_data + WRITE_PATH + f"{hpo_name}/optuna.log")
wr.s3.upload(local_file=f"optuna_objective_vs_gap_overfitting.pdf", path=s3_data + WRITE_PATH + f"{hpo_name}/optuna_objective_vs_gap_overfitting.pdf")

# Model params


In [ ]:
fixed_xgb_params = {
    "eval_metric": EVAL_METRIC,
    "tree_method": "hist",
    "enable_categorical": True,
    "n_jobs": -1,
    "verbosity": 0,
}

## Params selected trials

In [ ]:
cond=trials_results["flag_feasible_tau"]==1
top_trials=trials_results[cond].sort_values(by=["objective_mean"], ascending=True).head(30)["number"].tolist()

In [ ]:
trial_nums_to_evaluate=top_trials

WEIGHT_PARAM_KEYS = {"w_scale", "w_power"}

def split_weight_params(params_dict):
    xgb_params = {k: v for k, v in params_dict.items() if k not in WEIGHT_PARAM_KEYS}
    weight_params = {k: v for k, v in params_dict.items() if k in WEIGHT_PARAM_KEYS} or None
    return xgb_params, weight_params

model_params = {}
model_weight_params = {}

best_xgb, best_w = split_weight_params(study.best_params)
model_params["xgb_optuna_best_params"] = {**fixed_xgb_params, **best_xgb}
model_weight_params["xgb_optuna_best_params"] = best_w

for trial_num in trial_nums_to_evaluate:
    trial = study.trials[trial_num]
    if trial.state.name != "COMPLETE":
        print(f"Trial {trial_num} ignorado: estado {trial.state.name}")
        continue

    xgb_p, w_p = split_weight_params(trial.params)
    name_model = f"xgb_optuna_trial{trial_num}"
    model_params[name_model] = {**fixed_xgb_params, **xgb_p}
    model_weight_params[name_model] = w_p

# Hold-out validation performance
- Se entrena modelo con toda la muestra train hpo utilizada para encontrar los hiperparametros
- Se evalua la performance sobre un conjunto de validacion

In [ ]:
# Hold-out validation performance
holdout_models = {}
holdout_estimators = {}

output_dir = "."
os.makedirs(output_dir, exist_ok=True)

all_trials_pdf_name = f"{hpo_name}_holdout_all_trials_real_vs_pred.pdf"
all_trials_pdf_path = os.path.join(output_dir, all_trials_pdf_name)

with PdfPages(all_trials_pdf_path) as pdf:

    for name_model, params in model_params.items():
        print(f"Evaluando holdout {name_model}")

        fit_kwargs = {"verbose": False}
        weight_params = model_weight_params.get(name_model)
        if weight_params is not None:
            target_distance = np.abs(y_hpo_train.to_numpy() - 0.5)
            fit_kwargs["sample_weight"] = (
                1
                + weight_params["w_scale"]
                * target_distance ** weight_params["w_power"]
            )

        model = XGBRegressor(**params)
        model.fit(X_hpo_train, y_hpo_train, **fit_kwargs)

        holdout_results = summarize_model_results(
            estimator=model,
            X_train=X_hpo_train,
            y_train=y_hpo_train,
            X_validation=X_validation_holdout,
            y_validation=y_validation_holdout,
            selected_features=selected_features,
            name_model=name_model,
            overfitting_thresholds=(0.02, 0.05),
            success_threshold=SUCCESS_THRESHOLD,
        )

        holdout_models[name_model] = holdout_results
        holdout_estimators[name_model] = model

        df_pred = holdout_results["model_predictions"]

        for sample_name, sample_value in {
            "train_hpo": "train",
            "val_hpo": "validation",
        }.items():
            add_real_vs_pred_pages_to_pdf(
                df_pred=df_pred.loc[df_pred["sample"].eq(sample_value)],
                target_col="y_true",
                pred_col="y_pred",
                sample_name=sample_name,
                model_name=name_model,
                pdf=pdf,
                bins=50,
                show_plots=False,
                xlim=(0, 1),
                ylim=(0, 1),
                title_metrics=("mae", "rmse", "smape"),
                success_threshold=SUCCESS_THRESHOLD,
            )

        model_params_local_path = f"{name_model}_model_params.txt"
        with open(model_params_local_path, "w", encoding="utf-8") as f:
            f.write(json.dumps(params, indent=4, sort_keys=True, default=str))
            f.write("\n")

        wr.s3.upload(
            local_file=model_params_local_path,
            path=s3_data + WRITE_PATH + f"{hpo_name}/{name_model}/model_params.txt",
        )

        # Serializar pesos para reproducir el reentrenamiento futuro
        if weight_params is not None:
            weight_params_local_path = f"{name_model}_weight_params.txt"
            with open(weight_params_local_path, "w", encoding="utf-8") as f:
                f.write(json.dumps(weight_params, indent=4, sort_keys=True))
                f.write("\n")

            wr.s3.upload(
                local_file=weight_params_local_path,
                path=s3_data + WRITE_PATH + f"{hpo_name}/{name_model}/weight_params.txt",
            )

        gen_connector.write_file(
            df_pred,
            target=WRITE_PATH + f"{hpo_name}/{name_model}/df_val_holdout_pred",
            format_type="parquet",
            mode="overwrite",
        )

        for file_name, df_results in {
            "holdout_importances": holdout_results["feature_importances"],
            "holdout_model_summary": holdout_results["model_summary"],
        }.items():
            gen_connector.write_file(
                df_results,
                target=WRITE_PATH + f"{hpo_name}/{name_model}/{file_name}.csv",
                format_type="csv",
                index=False,
            )

print(f"PDF único guardado en local: {all_trials_pdf_path}")

In [ ]:
holdout_model_comparison = pd.concat(
    [model["model_summary"] for model in holdout_models.values()],
    ignore_index=True,
)

holdout_model_comparison.sort_values(["mae_val_mean", "gap_mean"], ascending=[True, True], inplace=True)

In [ ]:
cols_show=["name_model","mae_val_mean","mae_train_mean","feature_importance_top1", "feature_name_top1", "overfitting_diagnosis"]
display(holdout_model_comparison[cols_show].sort_values(by=["feature_importance_top1"], ascending=True))

## Write

In [ ]:
wr.s3.upload(
    local_file=all_trials_pdf_path,
    path=s3_data + WRITE_PATH + f"{hpo_name}/{all_trials_pdf_name}",
)

In [ ]:
with open(f"vars_hpo_{hpo_name}.py", "w") as f:
    f.write("VARS_HPO = [\n")
    for col in selected_features:
        f.write(f"    {repr(col)},\n")
    f.write("]\n")

In [ ]:
gen_connector.write_file(
    holdout_model_comparison,
    target=WRITE_PATH + f"{hpo_name}/holdout_model_comparison.csv",
    format_type="csv",
    index=False,
)